In [1]:
import os

os.environ[
    "TF_USE_LEGACY_KERAS"
] = "1"

In [2]:
!pip install -q tensorflow-recommenders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 2.8 MB/s eta 0:00:00a 0:00:01


In [3]:
import pandas as pd
import pickle

import tensorflow as tf
import tensorflow_recommenders as tfrs

import numpy as np

2026-05-23 13:13:59.958135: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779542040.211801      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779542040.289770      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779542040.870457      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779542040.870511      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779542040.870514      57 computation_placer.cc:177] computation placer alr

In [4]:
training_df = pd.read_parquet(
    "/kaggle/input/datasets/selinparmar/train-ds/training_dataset.parquet"
)

with open(
    "/kaggle/input/datasets/selinparmar/feature-vocab/feature_vocab.pkl",
    "rb"
) as file:

    feature_vocab = pickle.load(file)

In [5]:
training_df[
    'customer_id'
] = (
    training_df[
        'customer_id'
    ].astype(str)
)

training_df[
    'article_id'
] = (
    training_df[
        'article_id'
    ].astype(str)
)

In [6]:
user_vocab = (
    feature_vocab[
        'user_vocab'
    ]
)

favorite_product_vocab = (
    feature_vocab[
        'favorite_product_vocab'
    ]
)

item_vocab = (
    feature_vocab[
        'item_vocab'
    ]
)

product_type_vocab = (
    feature_vocab[
        'product_type_vocab'
    ]
)

season_vocab = (
    feature_vocab[
        'season_vocab'
    ]
)

In [7]:
class QueryTower(
    tf.keras.Model
):

    def __init__(self):

        super().__init__()

        # Customer ID embedding
        self.customer_embedding = tf.keras.Sequential([

            tf.keras.layers.StringLookup(
                vocabulary=user_vocab,
                mask_token=None
            ),

            tf.keras.layers.Embedding(
                len(user_vocab) + 1,
                32
            )
        ])

        # Favorite product type embedding
        self.favorite_product_embedding = tf.keras.Sequential([

            tf.keras.layers.StringLookup(
                vocabulary=
                favorite_product_vocab,
                mask_token=None
            ),

            tf.keras.layers.Embedding(
                len(
                    favorite_product_vocab
                ) + 1,
                16
            )
        ])

        # Season embedding
        self.season_embedding = tf.keras.Sequential([

            tf.keras.layers.StringLookup(
                vocabulary=
                season_vocab,
                mask_token=None
            ),

            tf.keras.layers.Embedding(
                len(
                    season_vocab
                ) + 1,
                8
            )
        ])

        # Dense network
        self.dense_layers = tf.keras.Sequential([

            tf.keras.layers.Dense(
                128,
                activation='relu'
            ),

            tf.keras.layers.Dense(64)
        ])

    def call(self, features):

        customer_vector = (
            self.customer_embedding(
                features['customer_id']
            )
        )

        favorite_product_vector = (
            self.favorite_product_embedding(
                features[
                    'favorite_product_type'
                ]
            )
        )

        season_vector = (
            self.season_embedding(
                features['season']
            )
        )

        numeric_features = tf.stack([

            tf.cast(
                features['age'],
                tf.float32
            ),

            tf.cast(
                features[
                    'purchase_frequency'
                ],
                tf.float32
            ),

            tf.cast(
                features[
                    'avg_spend'
                ],
                tf.float32
            ),

            tf.cast(
                features[
                    'repeat_purchase_ratio'
                ],
                tf.float32
            )

        ], axis=1)

        return self.dense_layers(
            tf.concat([

                customer_vector,
                favorite_product_vector,
                season_vector,
                numeric_features

            ], axis=1)
        )

In [8]:
class CandidateTower(
    tf.keras.Model
):

    def __init__(self):

        super().__init__()

        # Article embedding
        self.article_embedding = (
            tf.keras.Sequential([

                tf.keras.layers.StringLookup(
                    vocabulary=item_vocab,
                    mask_token=None
                ),

                tf.keras.layers.Embedding(
                    len(item_vocab) + 1,
                    32
                )
            ])
        )

        # Product type embedding
        self.product_embedding = (
            tf.keras.Sequential([

                tf.keras.layers.StringLookup(
                    vocabulary=
                    product_type_vocab,
                    mask_token=None
                ),

                tf.keras.layers.Embedding(
                    len(
                        product_type_vocab
                    ) + 1,
                    16
                )
            ])
        )

        # Season embedding
        self.season_embedding = (
            tf.keras.Sequential([

                tf.keras.layers.StringLookup(
                    vocabulary=
                    season_vocab,
                    mask_token=None
                ),

                tf.keras.layers.Embedding(
                    len(
                        season_vocab
                    ) + 1,
                    8
                )
            ])
        )

        # Dense network
        self.dense_layers = (
            tf.keras.Sequential([

                tf.keras.layers.Dense(
                    128,
                    activation='relu'
                ),

                tf.keras.layers.Dense(
                    64
                )
            ])
        )

    def call(
        self,
        features
    ):

        article_vector = (
            self.article_embedding(
                features[
                    'article_id'
                ]
            )
        )

        product_vector = (
            self.product_embedding(
                features[
                    'product_type_name'
                ]
            )
        )

        season_vector = (
            self.season_embedding(
                features[
                    'season'
                ]
            )
        )

        numeric_features = tf.stack([

            tf.cast(
                features[
                    'purchase_count'
                ],
                tf.float32
            )

        ], axis=1)

        combined_features = (
            tf.concat([

                article_vector,
                product_vector,
                season_vector,
                numeric_features

            ], axis=1)
        )

        return (
            self.dense_layers(
                combined_features
            )
        )

In [9]:
#candidate dataset
candidate_dataset = (
    tf.data.Dataset
    .from_tensor_slices(
        {
            "article_id":
                training_df[
                    "article_id"
                ],

            "product_type_name":
                training_df[
                    "product_type_name"
                ],

            "season":
                training_df[
                    "season"
                ],

            "purchase_count":
                training_df[
                    "purchase_count"
                ]
                .astype(float)
        }
    )
    .batch(256)
)

2026-05-23 13:15:33.427181: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [10]:
class TwoTowerModel(
    tfrs.models.Model
):

    def __init__(self):

        super().__init__()

        self.query_model = (
            QueryTower()
        )

        self.candidate_model = (
            CandidateTower()
        )

        self.task = (
            tfrs.tasks.Retrieval(
                metrics=
                tfrs.metrics.FactorizedTopK(

                    candidates=
                    candidate_dataset.map(
                        self.candidate_model
                    )
                )
            )
        )

    def compute_loss(
        self,
        features,
        training=False
    ):

        query_embeddings = (
            self.query_model(
                features
            )
        )

        candidate_embeddings = (
            self.candidate_model(
                features
            )
        )

        return self.task(
            query_embeddings,
            candidate_embeddings
        )

In [11]:
model = (
    TwoTowerModel()
)

model.compile(
    optimizer=
    tf.keras.optimizers.Adagrad(
        learning_rate=0.1
    )
)

In [12]:
dummy_features = {

    key:
    tf.constant(
        training_df[
            key
        ].head(1)
    )

    for key in
    training_df.columns
}

model.query_model(
    dummy_features
)

model.candidate_model(
    dummy_features
)

print(
    "Model built!"
)

Model built!


In [13]:
model.load_weights(
    "/kaggle/input/datasets/selinparmar/two-two-final/two_tower_weights.weights.h5"
)

print(
    "Final model loaded!"
)

Final model loaded!


In [14]:
article_lookup = (

    training_df[

        [
            'article_id',
            'product_type_name'
        ]

    ]

    .drop_duplicates()
)

In [15]:
sample_customer = (
    training_df[
        'customer_id'
    ]
    .iloc[0]
)

sample_customer

'0bae744d492a1ee45460ef798a977b33ffa09f14f92cf6d7c1fa5ace60ccba54'

In [16]:
customer_features = (

    training_df[

        training_df[
            'customer_id'
        ]
        ==
        sample_customer

    ]

    .iloc[0]
)

In [17]:
query_input = {

    'customer_id':
        tf.constant(
            [
                customer_features[
                    'customer_id'
                ]
            ]
        ),

    'favorite_product_type':
        tf.constant(
            [
                customer_features[
                    'favorite_product_type'
                ]
            ]
        ),

    'season':
        tf.constant(
            [
                customer_features[
                    'season'
                ]
            ]
        ),

    'age':
        tf.constant(
            [
                customer_features[
                    'age'
                ]
            ],
            dtype=tf.float32
        ),

    'purchase_frequency':
        tf.constant(
            [
                customer_features[
                    'purchase_frequency'
                ]
            ],
            dtype=tf.float32
        ),

    'avg_spend':
        tf.constant(
            [
                customer_features[
                    'avg_spend'
                ]
            ],
            dtype=tf.float32
        ),

    'repeat_purchase_ratio':
        tf.constant(
            [
                customer_features[
                    'repeat_purchase_ratio'
                ]
            ],
            dtype=tf.float32
        )
}

In [18]:
user_embedding = (
    model.query_model(
        query_input
    )
)

all_items = (
    training_df[
        'article_id'
    ]
    .unique()
)

scores = []

for item in all_items[:1000]:

    item_row = (
        training_df[
            training_df[
                'article_id'
            ]
            ==
            item
        ]
        .iloc[0]
    )

    item_input = {

        'article_id':
            tf.constant(
                [item]
            ),

        'product_type_name':
            tf.constant(
                [
                    item_row[
                        'product_type_name'
                    ]
                ]
            ),

        'season':
            tf.constant(
                [
                    item_row[
                        'season'
                    ]
                ]
            ),

        'purchase_count':
            tf.constant(
                [
                    item_row[
                        'purchase_count'
                    ]
                ],
                dtype=tf.float32
            )
    }

    item_embedding = (
        model.candidate_model(
            item_input
        )
    )

    score = tf.reduce_sum(

        user_embedding
        *
        item_embedding

    ).numpy()

    scores.append(
        (
            item,
            score
        )
    )

In [19]:
top_items = sorted(

    scores,

    key=lambda x: x[1],

    reverse=True

)[:10]

top_items

[('758084002', np.float32(-1353.9996)),
 ('603582003', np.float32(-1363.686)),
 ('501616007', np.float32(-1364.0415)),
 ('741040001', np.float32(-1364.5374)),
 ('660599013', np.float32(-1365.4377)),
 ('727948001', np.float32(-1367.26)),
 ('541518004', np.float32(-1369.3638)),
 ('799190003', np.float32(-1369.6323)),
 ('632982059', np.float32(-1370.8331)),
 ('562245102', np.float32(-1370.8962))]

In [20]:
recommended_products = []

for item_id, score in top_items:

    product = (
        article_lookup[

            article_lookup[
                'article_id'
            ]
            ==
            item_id

        ]

        .iloc[0]
    )

    recommended_products.append({

        "article_id":
            item_id,

        "product_type":
            product[
                'product_type_name'
            ],

        "score":
            score
    })

pd.DataFrame(
    recommended_products
)

,article_id,product_type,score
0,758084002,Swimsuit,-1353.999634
1,603582003,Shorts,-1363.686035
2,501616007,Shirt,-1364.041504
3,741040001,Bra,-1364.537354
4,660599013,Cardigan,-1365.437744
5,727948001,Bikini top,-1367.260010
6,541518004,Bra,-1369.363770
7,799190003,Sweater,-1369.632324
8,632982059,Top,-1370.833130
9,562245102,Trousers,-1370.896240


In [21]:
print(
    customer_features[
        'favorite_product_type'
    ]
)

print(
    customer_features[
        'season'
    ]
)

print(
    customer_features[
        'age'
    ]
)

Blazer
Autumn
23.0


In [22]:
sample_customer = (
    training_df[
        'customer_id'
    ]
    .iloc[50]
)

In [24]:
customer_features = (

    training_df[

        training_df[
            'customer_id'
        ]
        ==
        sample_customer

    ]

    .iloc[0]
)

In [25]:
query_input = {

    'customer_id':
        tf.constant(
            [
                customer_features[
                    'customer_id'
                ]
            ]
        ),

    'favorite_product_type':
        tf.constant(
            [
                customer_features[
                    'favorite_product_type'
                ]
            ]
        ),

    'season':
        tf.constant(
            [
                customer_features[
                    'season'
                ]
            ]
        ),

    'age':
        tf.constant(
            [
                customer_features[
                    'age'
                ]
            ],
            dtype=tf.float32
        ),

    'purchase_frequency':
        tf.constant(
            [
                customer_features[
                    'purchase_frequency'
                ]
            ],
            dtype=tf.float32
        ),

    'avg_spend':
        tf.constant(
            [
                customer_features[
                    'avg_spend'
                ]
            ],
            dtype=tf.float32
        ),

    'repeat_purchase_ratio':
        tf.constant(
            [
                customer_features[
                    'repeat_purchase_ratio'
                ]
            ],
            dtype=tf.float32
        )
}

In [26]:
user_embedding = (
    model.query_model(
        query_input
    )
)

all_items = (
    training_df[
        'article_id'
    ]
    .unique()
)

scores = []

for item in all_items[:1000]:

    item_row = (
        training_df[
            training_df[
                'article_id'
            ]
            ==
            item
        ]
        .iloc[0]
    )

    item_input = {

        'article_id':
            tf.constant(
                [item]
            ),

        'product_type_name':
            tf.constant(
                [
                    item_row[
                        'product_type_name'
                    ]
                ]
            ),

        'season':
            tf.constant(
                [
                    item_row[
                        'season'
                    ]
                ]
            ),

        'purchase_count':
            tf.constant(
                [
                    item_row[
                        'purchase_count'
                    ]
                ],
                dtype=tf.float32
            )
    }

    item_embedding = (
        model.candidate_model(
            item_input
        )
    )

    score = tf.reduce_sum(

        user_embedding
        *
        item_embedding

    ).numpy()

    scores.append(
        (
            item,
            score
        )
    )

In [27]:
top_items = sorted(

    scores,

    key=lambda x: x[1],

    reverse=True

)[:10]

top_items

[('562245102', np.float32(-1019.1188)),
 ('741040001', np.float32(-1019.5349)),
 ('316085001', np.float32(-1023.24207)),
 ('758084002', np.float32(-1023.8098)),
 ('603582003', np.float32(-1024.4005)),
 ('501616007', np.float32(-1025.1716)),
 ('727948001', np.float32(-1026.8936)),
 ('796210008', np.float32(-1031.619)),
 ('790096002', np.float32(-1033.6112)),
 ('762063003', np.float32(-1034.4366))]

In [28]:
recommended_products = []

for item_id, score in top_items:

    product = (
        article_lookup[

            article_lookup[
                'article_id'
            ]
            ==
            item_id

        ]

        .iloc[0]
    )

    recommended_products.append({

        "article_id":
            item_id,

        "product_type":
            product[
                'product_type_name'
            ],

        "score":
            score
    })

pd.DataFrame(
    recommended_products
)

,article_id,product_type,score
0,562245102,Trousers,-1019.118774
1,741040001,Bra,-1019.534912
2,316085001,Polo shirt,-1023.242065
3,758084002,Swimsuit,-1023.809814
4,603582003,Shorts,-1024.400513
5,501616007,Shirt,-1025.171631
6,727948001,Bikini top,-1026.893555
7,796210008,Trousers,-1031.619019
8,790096002,Top,-1033.611206
9,762063003,Dress,-1034.436646
